In [15]:
import numpy as np

def compute_skier_orientation_features(keypoints_21):
    """
    Computes body angle, ski orientations, and grab distances from a [21, 3] frame keypoint array.
    keypoints_21: [21, (x, y, conf)]
    Returns: dict of derived physical metrics
    """
    kpts = keypoints_21[:, :2]
    confs = keypoints_21[:, 2]

    # --- 1. Body Orientation Angle (Spine Vector) ---
    l_shoulder, r_shoulder = kpts[5], kpts[6]
    l_hip, r_hip = kpts[11], kpts[12]

    shoulder_center = (l_shoulder + r_shoulder) / 2.0
    hip_center = (l_hip + r_hip) / 2.0
    spine_vec = shoulder_center - hip_center  # Vector pointing from hips to shoulders

    # Angle relative to vertical axis (in degrees, -180 to 180)
    body_angle = np.degrees(np.arctan2(spine_vec[0], -spine_vec[1]))

    # --- 2. Ski Angles & Relative Crossing Angle ---
    l_ski_tip, l_ski_tail = kpts[17], kpts[18]
    r_ski_tip, r_ski_tail = kpts[19], kpts[20]

    l_ski_vec = l_ski_tip - l_ski_tail
    r_ski_vec = r_ski_tip - r_ski_tail

    angle_l_ski = np.degrees(np.arctan2(l_ski_vec[1], l_ski_vec[0]))
    angle_r_ski = np.degrees(np.arctan2(r_ski_vec[1], r_ski_vec[0]))

    # Absolute crossing angle between the two skis
    dot_product = np.dot(l_ski_vec, r_ski_vec)
    norms = np.linalg.norm(l_ski_vec) * np.linalg.norm(r_ski_vec)
    if norms > 1e-6:
        cos_angle = np.clip(dot_product / norms, -1.0, 1.0)
        crossed_ski_angle = np.degrees(np.arccos(cos_angle))
    else:
        crossed_ski_angle = 0.0

    # --- 3. Grab Distances (Wrist to Ski Tip/Tail) ---
    l_wrist, r_wrist = kpts[9], kpts[10]
    left_hand_ski_dist = min(
        np.linalg.norm(l_wrist - l_ski_tip),
        np.linalg.norm(l_wrist - r_ski_tip)
    )
    right_hand_ski_dist = min(
        np.linalg.norm(r_wrist - l_ski_tip),
        np.linalg.norm(r_wrist - r_ski_tip)
    )

    return {
        "body_angle": body_angle,
        "l_ski_angle": angle_l_ski,
        "r_ski_angle": angle_r_ski,
        "crossed_ski_angle": crossed_ski_angle,
        "min_grab_dist": min(left_hand_ski_dist, right_hand_ski_dist)
    }

In [2]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

# ==============================================================================
# 0. CONFIGURATION & INPUT INTEGRITY CHECKS
# ==============================================================================

DATA_DIR = Path("C:/Users/franc/Downloads/AQA-7/AQA-7/Actions/ski_big_air")
OUTPUT_DIR = Path("roi_pose_dataset")

def resolve_video(stem, data_dir=DATA_DIR):
    """Return the first existing video file for a label stem, else None."""
    stem = str(stem)
    candidates = (
        data_dir / f"{stem}.mp4",
        data_dir / f"{stem.zfill(3)}.avi",
        data_dir / f"{stem}.avi",
    )
    return next((p for p in candidates if p.is_file()), None)

# Load and validate labels
labels = pd.read_csv(DATA_DIR / "labels_CV.csv", sep=",")
errors, warnings = [], []

# Check duplicate filenames
dupes = labels.loc[labels["filename"].astype(str).duplicated(), "filename"].tolist()
if dupes:
    errors.append(f"duplicate filenames in labels_CV.csv: {dupes}")

# Check valid score ranges
bad_targets = labels.loc[~labels["target"].astype(float).between(0, 101), ["filename", "target"]]
if len(bad_targets):
    errors.append(f"targets outside 0-101 range:\n{bad_targets.to_string(index=False)}")

# Resolve video paths
labels["video_path"] = labels["filename"].map(resolve_video)
missing = labels.loc[labels["video_path"].isna(), "filename"].tolist()
if missing:
    errors.append(f"labels with no matching .mp4/.avi file: {missing}")

# Unused files warning
used = {str(p.resolve()) for p in labels["video_path"].dropna()}
on_disk = {p for p in DATA_DIR.iterdir() if p.suffix.lower() in {".mp4", ".avi"}}
unused = sorted(p.name for p in on_disk if str(p.resolve()) not in used)
if unused:
    warnings.append(f"video files with no label row: {unused}")

print(f"labels_CV.csv : {len(labels)} rows, targets {labels['target'].min()}-{labels['target'].max()}")
for msg in warnings:
    print(f"[WARN] {msg}")

if errors:
    print(f"\n{len(errors)} blocking issue(s):")
    for msg in errors:
        print(f"  [ERROR] {msg}")
    raise ValueError("Fix the input issues above before building the dataset.")

print("All input integrity checks passed successfully.\n")


# ==============================================================================
# 1. DYNAMIC TRACKER & MOTION COMPENSATION MODULES
# ==============================================================================

class GlobalMotionCompensator:
    """Estimates camera movement (pan, tilt, zoom) via ORB optical flow."""
    def __init__(self):
        self.prev_gray = None
        self.detector = cv2.ORB_create(nfeatures=500)

    def compensate(self, frame):
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        if self.prev_gray is None:
            self.prev_gray = gray
            return np.eye(2, 3, dtype=np.float32)

        kp1, des1 = self.detector.detectAndCompute(self.prev_gray, None)
        kp2, des2 = self.detector.detectAndCompute(gray, None)

        if des1 is None or des2 is None or len(kp1) < 10 or len(kp2) < 10:
            self.prev_gray = gray
            return np.eye(2, 3, dtype=np.float32)

        matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
        matches = matcher.match(des1, des2)
        matches = sorted(matches, key=lambda x: x.distance)[:50]

        pts1 = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
        pts2 = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

        M, _ = cv2.estimateAffinePartial2D(pts1, pts2)
        self.prev_gray = gray
        return M if M is not None else np.eye(2, 3, dtype=np.float32)


class DynamicPersonTracker:
    """Tracks target skier across camera motion and performs automatic recovery."""
    def __init__(self, pose_model_path="yolov8n-pose.pt"):
        self.model = YOLO(pose_model_path)
        self.gmc = GlobalMotionCompensator()
        self.tracker = None
        self.bbox = None
        self.is_tracking = False

    def _create_csrt_tracker(self):
        """Robust instantiator for CSRT tracker across different OpenCV versions."""
        if hasattr(cv2, 'TrackerCSRT_create'):
            return cv2.TrackerCSRT_create()
        elif hasattr(cv2, 'legacy') and hasattr(cv2.legacy, 'TrackerCSRT_create'):
            return cv2.legacy.TrackerCSRT_create()
        elif hasattr(cv2, 'TrackerCSRT') and hasattr(cv2.TrackerCSRT, 'create'):
            return cv2.TrackerCSRT.create()
        else:
            raise AttributeError(
                "CSRT Tracker is unavailable. Make sure 'opencv-contrib-python' "
                "is installed correctly: pip install opencv-contrib-python"
            )

    def init_tracker(self, frame, bbox):
        self.bbox = bbox
        self.tracker = self._create_csrt_tracker()
        self.tracker.init(frame, tuple(map(int, bbox)))
        self.is_tracking = True

    def update(self, frame):
        if not self.is_tracking:
            return None, False

        # Apply camera motion shift
        M = self.gmc.compensate(frame)
        if M is not None:
            cx = self.bbox[0] + self.bbox[2] / 2.0
            cy = self.bbox[1] + self.bbox[3] / 2.0
            pts = np.array([[[cx, cy]]], dtype=np.float32)
            transformed = cv2.transform(pts, M)
            new_cx, new_cy = transformed[0][0]
            self.bbox = (new_cx - self.bbox[2] / 2.0, new_cy - self.bbox[3] / 2.0, self.bbox[2], self.bbox[3])

        success, tracked_box = self.tracker.update(frame)
        if success:
            self.bbox = tracked_box
            return self.bbox, True
        else:
            self.is_tracking = False
            return self.bbox, False

    def recover_target(self, frame, prev_bbox, iou_threshold=0.2):
        """Full-frame re-detection fallback when visual tracking drifts."""
        results = self.model(frame, verbose=False)[0]
        if results.boxes is None or len(results.boxes) == 0:
            return False

        best_iou, best_box = 0.0, None
        px, py, pw, ph = prev_bbox
        p_area = pw * ph

        for box in results.boxes.xywh.cpu().numpy():
            cx, cy, w, h = box
            bx, by = cx - w / 2.0, cy - h / 2.0

            inter_x1, inter_y1 = max(px, bx), max(py, by)
            inter_x2, inter_y2 = min(px + pw, bx + w), min(py + ph, by + h)
            inter_w, inter_h = max(0.0, inter_x2 - inter_x1), max(0.0, inter_y2 - inter_y1)
            inter_area = inter_w * inter_h
            union_area = p_area + (w * h) - inter_area
            iou = inter_area / union_area if union_area > 0 else 0

            if iou > best_iou and iou > iou_threshold:
                best_iou = iou
                best_box = (bx, by, w, h)

        if best_box is not None:
            self.init_tracker(frame, best_box)
            return True
        return False


def remap_crop_keypoints(kpts, scale, pad_x, pad_y, crop_x1, crop_y1):
    """Maps keypoint predictions from 640x640 cropped input back to video space."""
    if kpts is None or len(kpts) == 0:
        return np.array([])
    remapped = kpts.copy()
    remapped[..., 0] = (remapped[..., 0] - pad_x) / scale + crop_x1
    remapped[..., 1] = (remapped[..., 1] - pad_y) / scale + crop_y1
    return remapped


def find_skier_entry_frame(video_path, model, min_conf=0.6, max_scan_frames=150):
    """Scans video until the skier enters the frame."""
    cap = cv2.VideoCapture(str(video_path))
    frame_idx = 0

    while cap.isOpened() and frame_idx < max_scan_frames:
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, verbose=False)[0]
        if results.boxes is not None and len(results.boxes) > 0:
            persons = [b for b in results.boxes if int(b.cls[0]) == 0 and float(b.conf[0]) >= min_conf]
            if persons:
                best_person = max(persons, key=lambda b: float(b.conf[0]))
                cx, cy, w, h = best_person.xywh.cpu().numpy()[0]
                init_box = (cx - w / 2.0, cy - h / 2.0, w, h)
                cap.release()
                return frame_idx, frame, init_box
        frame_idx += 1

    cap.release()
    return None, None, None


# ==============================================================================
# 2. MAIN DATASET PROCESSING PIPELINE
# ==============================================================================

def process_aqa_dataset(labels_df, output_dir=OUTPUT_DIR):
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    pipeline = DynamicPersonTracker(pose_model_path="yolov8n-pose.pt")
    processed_records = []

    for idx, row in labels_df.iterrows():
        video_file = row["video_path"]
        target_score = row["target"]
        stem_name = Path(video_file).stem

        print(f"[{idx+1}/{len(labels_df)}] Processing: {stem_name} (Target Score: {target_score})")

        # Step 1: Scan for skier entry frame
        entry_idx, entry_frame, init_box = find_skier_entry_frame(video_file, pipeline.model)

        if init_box is None:
            print(f"  [SKIP] No skier detected in initial scan of {stem_name}")
            continue

        print(f"  -> Skier detected entering at Frame {entry_idx}")

        # Fill pre-entry frames with zeros
        video_keypoints_sequence = [np.zeros((17, 3), dtype=np.float32) for _ in range(entry_idx)]

        # Step 2: Initialize tracker at entry point
        pipeline.init_tracker(entry_frame, init_box)

        # Open video and fast-forward to entry frame
        cap = cv2.VideoCapture(str(video_file))
        cap.set(cv2.CAP_PROP_POS_FRAMES, entry_idx)

        # Step 3: Process video frames
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            bbox, tracking_ok = pipeline.update(frame)

            if not tracking_ok:
                re_detected = pipeline.recover_target(frame, pipeline.bbox)
                if re_detected:
                    bbox = pipeline.bbox
                    tracking_ok = True

            frame_kpts = np.zeros((17, 3), dtype=np.float32)

            if tracking_ok and bbox is not None:
                x, y, w, h = [int(v) for v in bbox]
                h_f, w_f = frame.shape[:2]

                # 15% ROI margin expansion
                margin = 0.15
                x1 = max(0, int(x - w * margin))
                y1 = max(0, int(y - h * margin))
                x2 = min(w_f, int(x + w * (1 + margin)))
                y2 = min(h_f, int(y + h * (1 + margin)))

                crop = frame[y1:y2, x1:x2]
                ch, cw = crop.shape[:2]

                if ch > 0 and cw > 0:
                    scale = min(640.0 / cw, 640.0 / ch)
                    nw, nh = int(cw * scale), int(ch * scale)
                    resized = cv2.resize(crop, (nw, nh))

                    padded = np.full((640, 640, 3), 114, dtype=np.uint8)
                    pad_x, pad_y = (640 - nw) // 2, (640 - nh) // 2
                    padded[pad_y:pad_y + nh, pad_x:pad_x + nw] = resized

                    res = pipeline.model(padded, verbose=False)[0]

                    if res.keypoints is not None and len(res.keypoints) > 0:
                        crop_kpts = res.keypoints.xy.cpu().numpy()[0]
                        confs = res.keypoints.conf.cpu().numpy()[0]

                        remapped = remap_crop_keypoints(crop_kpts[None, ...], scale, pad_x, pad_y, x1, y1)[0]
                        frame_kpts[:, :2] = remapped
                        frame_kpts[:, 2] = confs

            video_keypoints_sequence.append(frame_kpts)

        cap.release()

        # Step 4: Export arrays
        if len(video_keypoints_sequence) > 0:
            seq_array = np.array(video_keypoints_sequence, dtype=np.float32)  # Shape: [T, 17, 3]
            save_file = output_path / f"{stem_name}_pose.npy"
            np.save(save_file, seq_array)

            processed_records.append({
                "filename": row["filename"],
                "pose_feature_path": str(save_file),
                "num_frames": len(seq_array),
                "target": target_score
            })

    dataset_summary = pd.DataFrame(processed_records)
    summary_path = output_path / "dataset_processed_features.csv"
    dataset_summary.to_csv(summary_path, index=False)
    print(f"\nPipeline finished successfully! Features saved to: {summary_path}")
    return dataset_summary


# Execute complete pipeline
dataset_features_df = process_aqa_dataset(labels)

labels_CV.csv : 234 rows, targets 0.0-100.0
All input integrity checks passed successfully.

[1/234] Processing: 001 (Target Score: 76.20881385)
  -> Skier detected entering at Frame 49
[2/234] Processing: 002 (Target Score: 86.79272055)
  [SKIP] No skier detected in initial scan of 002
[3/234] Processing: 003 (Target Score: 74.20904845)
  -> Skier detected entering at Frame 21
[4/234] Processing: 004 (Target Score: 87.62160681)
  -> Skier detected entering at Frame 15
[5/234] Processing: 005 (Target Score: 87.62160681)
  -> Skier detected entering at Frame 81
[6/234] Processing: 006 (Target Score: 76.20881385)
  -> Skier detected entering at Frame 36
[7/234] Processing: 007 (Target Score: 81.10001083)
  [SKIP] No skier detected in initial scan of 007
[8/234] Processing: 008 (Target Score: 85.99120781)
  -> Skier detected entering at Frame 43
[9/234] Processing: 009 (Target Score: 76.20881385)
  -> Skier detected entering at Frame 39
[10/234] Processing: 010 (Target Score: 10.96256995)

In [14]:
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO

# COCO Keypoint Skeleton Connections for visualization
SKELETON_CONNECTIONS = [
    (0, 1), (0, 2), (1, 3), (2, 4),             # Head (Nose, Eyes, Ears)
    (5, 6),                                     # Shoulders
    (5, 7), (7, 9), (6, 8), (8, 10),            # Arms
    (5, 11), (6, 12), (11, 12),                 # Torso
    (11, 13), (13, 15), (12, 14), (14, 16)      # Legs
]


def draw_skeleton(frame, keypoints, confidence_threshold=0.3):
    """Draws keypoints and skeleton lines directly on the full frame."""
    if keypoints is None or len(keypoints) == 0:
        return frame

    kpts = keypoints[0]  # First detected person

    # Draw connection lines (bones)
    for p1, p2 in SKELETON_CONNECTIONS:
        x1, y1, c1 = kpts[p1]
        x2, y2, c2 = kpts[p2]

        if c1 >= confidence_threshold and c2 >= confidence_threshold:
            pt1 = (int(x1), int(y1))
            pt2 = (int(x2), int(y2))
            cv2.line(frame, pt1, pt2, (0, 255, 255), 2)  # Yellow bones

    # Draw joint points
    for x, y, conf in kpts:
        if conf >= confidence_threshold:
            cv2.circle(frame, (int(x), int(y)), 5, (0, 0, 255), -1)  # Red joints
            cv2.circle(frame, (int(x), int(y)), 2, (255, 255, 255), -1)

    return frame


def visualize_single_video_pipeline(labels_df, video_index=0):
    """
    Runs the pipeline on a single video and displays real-time visual feedback:
      - Blue Box: Active tracking ROI
      - Red/Yellow Skeleton: Remapped YOLO pose keypoints
      - Top-Left Inset: Cropped 640x640 model input
    """
    if labels_df.empty:
        print("[ERROR] labels DataFrame is empty.")
        return

    sample_row = labels_df.iloc[video_index]
    video_path = sample_row["video_path"]
    target_score = sample_row["target"]
    stem_name = Path(video_path).stem

    print(f"\n--- Visualizing Video [{video_index + 1}/{len(labels_df)}]: {stem_name} ---")
    print(f"File Path: {video_path}")
    print(f"Ground Truth Score: {target_score}")

    pipeline = DynamicPersonTracker(pose_model_path="yolov8n-pose.pt")

    # Step 1: Entry scan
    entry_idx, entry_frame, init_box = find_skier_entry_frame(video_path, pipeline.model)

    if init_box is None:
        print(f"[FAIL] No skier detected during entry scan of {stem_name}")
        return

    print(f"[SUCCESS] Skier detected entering at Frame {entry_idx}")
    pipeline.init_tracker(entry_frame, init_box)

    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, entry_idx)

    cv2.namedWindow(f"YOLO Pose Visual Check - {stem_name}", cv2.WINDOW_NORMAL)

    frame_count = entry_idx
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        bbox, tracking_ok = pipeline.update(frame)

        if not tracking_ok:
            re_detected = pipeline.recover_target(frame, pipeline.bbox)
            if re_detected:
                bbox = pipeline.bbox
                tracking_ok = True

        display_frame = frame.copy()
        crop_inset = np.zeros((160, 160, 3), dtype=np.uint8)

        if tracking_ok and bbox is not None:
            x, y, w, h = [int(v) for v in bbox]
            h_f, w_f = frame.shape[:2]

            # 15% ROI expansion
            margin = 0.15
            x1 = max(0, int(x - w * margin))
            y1 = max(0, int(y - h * margin))
            x2 = min(w_f, int(x + w * (1 + margin)))
            y2 = min(h_f, int(y + h * (1 + margin)))

            crop = frame[y1:y2, x1:x2]
            ch, cw = crop.shape[:2]

            if ch > 0 and cw > 0:
                scale = min(640.0 / cw, 640.0 / ch)
                nw, nh = int(cw * scale), int(ch * scale)
                resized = cv2.resize(crop, (nw, nh))

                padded = np.full((640, 640, 3), 114, dtype=np.uint8)
                pad_x, pad_y = (640 - nw) // 2, (640 - nh) // 2
                padded[pad_y:pad_y + nh, pad_x:pad_x + nw] = resized

                crop_inset = cv2.resize(padded, (160, 160))

                # YOLO Pose inference
                res = pipeline.model(padded, verbose=False)[0]

                if res.keypoints is not None and len(res.keypoints) > 0:
                    crop_kpts = res.keypoints.xy.cpu().numpy()[0]
                    confs = res.keypoints.conf.cpu().numpy()[0]

                    # Remap keypoints to full video
                    remapped = remap_crop_keypoints(crop_kpts[None, ...], scale, pad_x, pad_y, x1, y1)
                    kpts_with_conf = np.dstack([remapped, confs[None, ...]])

                    # Draw skeleton
                    display_frame = draw_skeleton(display_frame, kpts_with_conf)

            # Draw blue tracking box
            cv2.rectangle(display_frame, (x1, y1), (x2, y2), (255, 128, 0), 2)
            cv2.putText(display_frame, f"Tracked ROI ({w}x{h})", (x1, max(20, y1 - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 128, 0), 2)

        else:
            cv2.putText(display_frame, "STATUS: TARGET LOST", (30, 80),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

        # Overlay crop inset thumbnail in top-left corner
        display_frame[10:170, 10:170] = crop_inset
        cv2.rectangle(display_frame, (10, 10), (170, 170), (255, 255, 255), 1)
        cv2.putText(display_frame, "Model Input Crop", (15, 165),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

        # Draw frame counter and score info
        cv2.putText(display_frame, f"Frame: {frame_count} | Target Score: {target_score}", (180, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        cv2.imshow(f"YOLO Pose Visual Check - {stem_name}", display_frame)

        # Press 'q' or 'ESC' to quit, spacebar to pause
        key = cv2.waitKey(30) & 0xFF
        if key == ord('q') or key == 27:
            break
        elif key == ord(' '):  # Pause on spacebar
            cv2.waitKey(0)

        frame_count += 1

    cap.release()
    cv2.destroyAllWindows()


# --- Run visual verification on video index 0 ---
visualize_single_video_pipeline(labels, video_index=225)


--- Visualizing Video [226/234]: 1026 (1)(2)-51 ---
File Path: C:\Users\franc\Downloads\AQA-7\AQA-7\Actions\ski_big_air\1026 (1)(2)-51.mp4
Ground Truth Score: 26.146382
[SUCCESS] Skier detected entering at Frame 13
